In [ ]:
# Step 1: Install Required Dependencies
%pip install --upgrade pip

# Uninstall conflicting packages
%pip uninstall -y langchain_classic langchain-core langchain-openai langchain-community langchain langchain-chroma chromadb beautifulsoup4 python-dotenv PyPDF2 rank_bm25 weaviate-client ragas wikipedia langchain-weaviate langchain-together langchain-experimental tiktoken langgraph langchain-tavily

%pip install langchain==1.1.0
%pip install langgraph==1.0.4
%pip install langchain-openai==1.1.0
%pip install langchain-chroma==1.0.0
%pip install chromadb==1.3.5
%pip install python-dotenv==1.2.1

In [ ]:
# PRE-STEP: Build the Basic Agent
# Step 2: Import Core Libraries and Configure Environment
import os
os.environ['USER_AGENT'] = 'RAGUserAgent'
import openai
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
import chromadb
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END

# Load environment variables
_ = load_dotenv(dotenv_path='env.txt')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
openai.api_key = os.environ['OPENAI_API_KEY']

# Initialize models
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()

# Define Agent State with memory placeholders
class AgentState(TypedDict):
    """State container for agent memory and messages"""
    messages: Annotated[Sequence[BaseMessage], add_messages]
    working_memory: dict  # Short-term context
    episodic_recall: list  # Retrieved past experiences
    semantic_facts: dict  # Retrieved knowledge

# Initialize vector store for future memory storage
vector_store = Chroma(
    collection_name="agent_memory",
    embedding_function=embeddings,
    persist_directory="./memory_store"
)

# Create base prompt and chain
base_prompt = PromptTemplate.from_template("""
You are a helpful assistant with memory capabilities.

Current conversation:
{messages}

Please respond to the latest message.
""")

output_parser = StrOutputParser()

# Define agent node
def agent_node(state: AgentState) -> dict:
    """Core agent logic - processes messages and generates responses"""
    messages = state["messages"][-5:] if state["messages"] else []
    formatted_messages = "\n".join([
        f"{msg.type}: {msg.content}" 
        for msg in messages
    ])
    
    chain = base_prompt | llm | output_parser
    response = chain.invoke({"messages": formatted_messages})
    
    return {"messages": [("assistant", response)]}

# Build and compile the graph
workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.set_entry_point("agent")
workflow.add_edge("agent", END)

app = workflow.compile()

In [ ]:
# PRE-STEP: Test the agent
test_input = {
    "messages": [("user", "Hello! What's the capital of France?")]
}

result = app.invoke(test_input)
print(result["messages"][-1].content)

In [ ]:
# Step 1: Import Additional Dependencies for Episodic Memory
from datetime import datetime
from typing import List
from langchain_core.documents import Document

In [ ]:
# Step 2: Create Episodic Memory Storage Functions
def store_episodic_memory(vector_store, conversation_id: str, messages: List, summary: str = None):
    """Store a conversation episode in vector memory"""
    if not summary and messages:
        summary = f"Conversation about: {messages[0].content[:100]}..."
    
    doc = Document(
        page_content="\n".join([f"{msg.type}: {msg.content}" for msg in messages]),
        metadata={
            "type": "episodic",
            "conversation_id": conversation_id,
            "timestamp": datetime.now().isoformat(),
            "message_count": len(messages)
        }
    )
    vector_store.add_documents([doc])
    return conversation_id

def retrieve_episodic_memories(vector_store, query: str, k: int = 3):
    """Retrieve relevant past conversation episodes"""
    return vector_store.similarity_search(query, k=k, filter={"type": "episodic"})

In [ ]:
# Step 3: Enhance Agent Node with Episodic Recall
def agent_with_episodic_memory(state: AgentState) -> dict:
    """Agent that retrieves and uses episodic memories"""
    messages = state.get("messages", [])
    
    # Retrieve relevant memories
    past_episodes = retrieve_episodic_memories(vector_store, messages[-1].content, k=2) if messages else []
    episodic_context = ("Relevant past conversations:\n" + "\n".join(
        f"\n[{ep.metadata.get('timestamp', 'Unknown')}]:\n{ep.page_content[:200]}..."
        for ep in past_episodes
    )) if past_episodes else ""
    
    # Generate response
    response = (PromptTemplate.from_template("""
You are a helpful assistant with episodic memory of past conversations.

{episodic_context}

Current conversation:
{messages}

Please respond to the latest message, utilizing relevant past conversations if helpful.
""") | llm | output_parser).invoke({
        "episodic_context": episodic_context,
        "messages": "\n".join(f"{m.type}: {m.content}" for m in messages[-5:]) if messages else ""
    })
    
    return {"messages": [("assistant", response)], "episodic_recall": past_episodes}

In [ ]:
# Step 4: Build Memory-Enabled Workflow
def store_conversation_node(state: AgentState) -> dict:
    """Store the current conversation as an episodic memory"""
    messages = state.get("messages", [])
    if len(messages) >= 2:  # Only store meaningful conversations
        store_episodic_memory(vector_store, f"conv_{datetime.now().timestamp()}", messages)
    return {"working_memory": {"stored": True}}

# Create workflow with episodic memory
memory_workflow = StateGraph(AgentState)
memory_workflow.add_node("recall_and_respond", agent_with_episodic_memory)
memory_workflow.add_node("store_memory", store_conversation_node)
memory_workflow.set_entry_point("recall_and_respond")
memory_workflow.add_edge("recall_and_respond", "store_memory")
memory_workflow.add_edge("store_memory", END)

memory_app = memory_workflow.compile()

In [ ]:
# Step 5: Test Episodic Memory Functionality
# First conversation - store a memory
result_1 = memory_app.invoke({
    "messages": [("user", "I'm planning a trip to Paris next month. Any recommendations?")]
})
print("First conversation response:")
print(result_1["messages"][-1].content)
print("\n" + "="*50 + "\n")

# Second conversation - should recall the Paris discussion
result_2 = memory_app.invoke({
    "messages": [("user", "What are some good restaurants in Paris?")]
})
print("Second conversation response (with episodic recall):")
print(result_2["messages"][-1].content)

if result_2.get("episodic_recall"):
    print(f"\nRecalled {len(result_2['episodic_recall'])} relevant memories")